# Unit 1 – What are Tools?

**Hugging Face AI Agents Course** · Unidad 1, sección *What are Tools?*

En esta sección el curso explica qué es una herramienta (*tool*), qué información necesita un LLM para usarla y cómo se le entrega esa información a través del *system prompt*. En este notebook reproduzco el ejemplo de la calculadora paso a paso y, al final, lo pruebo con un LLM real (Gemini) para ver el ciclo completo de una llamada a herramienta.

## 1. Una herramienta es una función con un objetivo claro

Empiezo con la función más simple del curso: una calculadora que multiplica dos enteros. Tiene un nombre descriptivo, un *docstring* que explica qué hace y *type hints* en las entradas y la salida.

In [1]:
def calculator(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

calculator(6, 7)

42

## 2. Descripción textual de la herramienta (a mano)

El LLM no ve el código, solo texto. Por eso hay que describirle la herramienta: **qué hace**, **qué entradas espera** y **qué devuelve**. Primero lo escribo manualmente:

In [2]:
manual_description = "Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int"
print(manual_description)

Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int


## 3. Clase genérica `Tool`

Escribir las descripciones a mano es frágil (es fácil olvidar un detalle o cambiar el formato entre herramientas). La clase `Tool` guarda todo en un solo objeto y genera la descripción con `to_string()`. Además, con `__call__` el objeto se puede llamar como si fuera la función.

In [3]:
from typing import Callable


class Tool:
    """
    A class representing a reusable piece of code (Tool).

    Attributes:
        name (str): Name of the tool.
        description (str): A textual description of what the tool does.
        func (callable): The function this tool wraps.
        arguments (list): A list of arguments.
        outputs (str or list): The return type(s) of the wrapped function.
    """
    def __init__(self,
                 name: str,
                 description: str,
                 func: Callable,
                 arguments: list,
                 outputs: str):
        self.name = name
        self.description = description
        self.func = func
        self.arguments = arguments
        self.outputs = outputs

    def to_string(self) -> str:
        """
        Return a string representation of the tool,
        including its name, description, arguments, and outputs.
        """
        args_str = ", ".join([
            f"{arg_name}: {arg_type}" for arg_name, arg_type in self.arguments
        ])

        return (
            f"Tool Name: {self.name},"
            f" Description: {self.description},"
            f" Arguments: {args_str},"
            f" Outputs: {self.outputs}"
        )

    def __call__(self, *args, **kwargs):
        """
        Invoke the underlying function (callable) with provided arguments.
        """
        return self.func(*args, **kwargs)

Creo la herramienta pasando cada dato a mano y compruebo que la descripción generada es igual a la que escribí en el paso 2:

In [4]:
calculator_tool = Tool(
    "calculator",                   # name
    "Multiply two integers.",       # description
    calculator,                     # function to call
    [("a", "int"), ("b", "int")],   # inputs (names and types)
    "int",                          # output
)

print(calculator_tool.to_string())
print("¿Igual a la descripción manual?", calculator_tool.to_string() == manual_description)
print("calculator_tool(6, 7) =", calculator_tool(6, 7))

Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int
¿Igual a la descripción manual? True
calculator_tool(6, 7) = 42


## 4. Decorador `@tool`: la descripción sale sola del código

El código de la función ya tiene todo lo necesario: el nombre (`calculator`), la descripción (el *docstring*) y los tipos de entrada y salida (*type hints*). El decorador `@tool` usa el módulo `inspect` de Python para leer esa información y construir el objeto `Tool` automáticamente.

In [5]:
import inspect


def tool(func):
    """
    A decorator that creates a Tool instance from the given function.
    """
    # Get the function signature
    signature = inspect.signature(func)

    # Extract (param_name, param_annotation) pairs for inputs
    arguments = []
    for param in signature.parameters.values():
        annotation_name = (
            param.annotation.__name__
            if hasattr(param.annotation, '__name__')
            else str(param.annotation)
        )
        arguments.append((param.name, annotation_name))

    # Determine the return annotation
    return_annotation = signature.return_annotation
    if return_annotation is inspect._empty:
        outputs = "No return annotation"
    else:
        outputs = (
            return_annotation.__name__
            if hasattr(return_annotation, '__name__')
            else str(return_annotation)
        )

    # Use the function's docstring as the description (default if None)
    description = func.__doc__ or "No description provided."

    # The function name becomes the Tool name
    name = func.__name__

    # Return a new Tool instance
    return Tool(
        name=name,
        description=description,
        func=func,
        arguments=arguments,
        outputs=outputs
    )

In [6]:
@tool
def calculator(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

print(calculator.to_string())
print("calculator(6, 7) =", calculator(6, 7))

Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int
calculator(6, 7) = 42


Con el decorador se obtiene exactamente la misma descripción, pero sin escribirla a mano. Para comprobar que sirve con cualquier función, defino una segunda herramienta (la del clima, como en el notebook *dummy agent*):

In [7]:
@tool
def get_weather(location: str) -> str:
    """Get the current weather in a given location."""
    return f"the weather in {location} is sunny with low temperatures."

print(get_weather.to_string())

Tool Name: get_weather, Description: Get the current weather in a given location., Arguments: location: str, Outputs: str


## 5. Inyectar las herramientas en el *system prompt*

Así es como el LLM se entera de que existen las herramientas: las descripciones generadas con `to_string()` se insertan en el *system prompt*, junto con las instrucciones de cómo pedir una herramienta (un JSON con `action` y `action_input`).

In [8]:
tools = [calculator, get_weather]
tools_description = "\n".join(t.to_string() for t in tools)

SYSTEM_PROMPT = """You are an AI assistant designed to help users efficiently and accurately.
You have access to the following tools:

{tools_description}

To use a tool, write a JSON blob with an `action` key (the tool name) and an
`action_input` key (a dictionary with the arguments). Use only ONE action at a time:

Thought: think about what to do
Action:
```json
{"action": "<tool name>", "action_input": {"<arg>": <value>}}
```
Observation: the result of the tool (this is provided to you, never invent it)

When you know the answer, finish with:
Thought: I now know the final answer
Final Answer: the final answer to the original question
""".replace("{tools_description}", tools_description)

print(SYSTEM_PROMPT)

You are an AI assistant designed to help users efficiently and accurately.
You have access to the following tools:

Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int
Tool Name: get_weather, Description: Get the current weather in a given location., Arguments: location: str, Outputs: str

To use a tool, write a JSON blob with an `action` key (the tool name) and an
`action_input` key (a dictionary with the arguments). Use only ONE action at a time:

Thought: think about what to do
Action:
```json
{"action": "<tool name>", "action_input": {"<arg>": <value>}}
```
Observation: the result of the tool (this is provided to you, never invent it)

When you know the answer, finish with:
Thought: I now know the final answer
Final Answer: the final answer to the original question



## 6. Prueba con un LLM real (Gemini)

Para ver que esto funciona de verdad, le hago al modelo una multiplicación grande. Uso Gemini (gratis) en vez de la API de Hugging Face, igual que en el notebook *dummy agent*. La llave está guardada en los *Secrets* de Colab con el nombre `GEMINI_API_KEY`.

In [9]:
!pip install -q openai

In [10]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    timeout=60,
    max_retries=0,
)
MODEL = "gemini-4-hydrogen"

**Paso A – el LLM decide qué herramienta usar.** Corto la generación en `Observation:` para que el modelo no se invente el resultado.

In [11]:
question = "What is 1234 multiplied by 5678?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": question},
]

output = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    max_tokens=300,
    stop=["Observation:"],
    reasoning_effort="low",
)
llm_action = output.choices[0].message.content
print(llm_action)

Action:
```json
{"action": "calculator", "action_input": {"a": 1234, "b": 5678}}
```Final Answer: 7006652


**Paso B – el agente (mi código) ejecuta la herramienta.** El LLM solo escribió texto; aquí leo el JSON, busco la herramienta por su nombre y la ejecuto con los argumentos que pidió.

In [12]:
import json
import re

tools_by_name = {t.name: t for t in tools}

json_text = re.search(r"\{.*\}", llm_action, re.DOTALL).group(0)
call = json.loads(json_text)
print("Herramienta pedida:", call["action"])
print("Argumentos:", call["action_input"])

observation = tools_by_name[call["action"]](**call["action_input"])
print("Observation:", observation)

Herramienta pedida: calculator
Argumentos: {'a': 1234, 'b': 5678}
Observation: 7006652


**Paso C – le devuelvo la observación al LLM para que dé la respuesta final.** Gemini no permite que la conversación termine con un mensaje del asistente, así que agrego un mensaje de usuario pidiéndole que continúe.

In [13]:
messages += [
    {"role": "assistant", "content": llm_action + f"Observation: {observation}\n"},
    {"role": "user", "content": "Continue from the Observation above and give the Final Answer."},
]

output = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    max_tokens=200,
    reasoning_effort="low",
)
print(output.choices[0].message.content)
print("\nComprobación en Python: 1234 * 5678 =", 1234 * 5678)

Final Answer: 1234 multiplied by 5678 is 7006652.

Comprobación en Python: 1234 * 5678 = 7006652


## Lo que entendí

- Una **herramienta** es una función que complementa al LLM en lo que no hace bien solo: cálculos exactos, datos actualizados, consultas a APIs, etc.
- El LLM **no ejecuta** la herramienta. Solo recibe una **descripción en texto** (nombre, qué hace, argumentos con tipos y salida) dentro del *system prompt*, y cuando la necesita escribe un texto que la "pide".
- Quien ejecuta la herramienta es el **agente** (mi código): lee el JSON, llama a la función y le devuelve el resultado al modelo como una *Observation*.
- Escribir las descripciones a mano es frágil. Con la clase `Tool` y el decorador `@tool` la descripción se genera sola a partir del nombre, el *docstring* y los *type hints* de la función, que es lo mismo que hacen librerías como smolagents.
- **MCP** (*Model Context Protocol*) lleva esta idea más lejos: es un estándar para que las mismas herramientas sirvan con cualquier LLM o framework sin reimplementarlas.